# 🎥 Shorts Creator IA: Automated Podcast Highlights

Este notebook automatiza a criação de cortes virais para podcasts usando Inteligência Artificial. 

**Hardware Sugerido:** GPU T4 (ou superior) no Google Colab.

## 1. Setup do Ambiente (Conda)
Seguindo as instruções do projeto, vamos preparar o ambiente utilizando o `conda` dentro do Colab.

In [ ]:
# @title 🛠️ Instalar Conda e Dependências
!pip install -q condacolab
import condacolab
condacolab.install()

print("Aguarde o kernel reiniciar e siga para a próxima célula.")

In [ ]:
# @title 🚀 Clonar Projeto e Criar Ambiente
import os
!git clone https://github.com/armandocastrodesousajunior/shorts-creator-ia.git
%cd shorts-creator-ia

# Instala as dependências do environment.yml no ambiente base do Conda Colab
!mamba env update -n base -f environment.yml
!apt-get install -y ffmpeg

print("Ambiente preparado com sucesso!")

## 2. Upload do Vídeo
Use esta célula para carregar o seu podcast (.mp4) diretamente para o ambiente.

In [ ]:
# @title 📥 Upload do Podcast (.mp4)
from google.colab import files
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"Vídeo carregado: {video_filename}")

## 3. Configuração e Inicialização

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from transcription import AudioTranscriber
from visual_analysis import VisualAnalyzer
from moment_selection import MomentSelector
from video_editor import VideoEditor
import torch

# @title Parâmetros de Execução
MODE = "Fast" # @param ["Fast", "High Quality"]
WHISPER_MODEL = "large-v3" # @param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2" # @param ["mistralai/Mistral-7B-Instruct-v0.2", "meta-llama/Meta-Llama-3-8B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]
LLAVA_MODEL = "llava-hf/llava-1.5-7b-hf"

print("Inicializando modelos...")
transcriber = AudioTranscriber(model_size=WHISPER_MODEL)
selector = MomentSelector(model_id=LLM_MODEL)
editor = VideoEditor()

## 4. Executar Pipeline

In [ ]:
# @title 🎬 Gerar Cortes Virais
output_dir = "saida_cortes"
os.makedirs(output_dir, exist_ok=True)

# 1. Transcrição
print("Passo 1: Transcrevendo áudio...")
transcription = transcriber.transcribe(video_filename)

# 2. Momentos
print("Passo 2: Analisando melhores momentos...")
moments = selector.select_moments(transcription)
print(f"Identificados {len(moments)} momentos interessantes.")

# 3. Geração de Clipes e JSONs
print("Passo 3: Exportando vídeos e metadados...")
clip_paths = []
import json

for i, m in enumerate(moments):
    clip_name = f"clip_{i}"
    mp4_path = os.path.join(output_dir, f"{clip_name}.mp4")
    json_path = os.path.join(output_dir, f"{clip_name}.json")
    
    if editor.cut_video(video_filename, mp4_path, m['start'], m['end']):
        clip_paths.append(mp4_path)
        meta = {
            "clip_id": i,
            "start": m['start'],
            "end": m['end'],
            "reason": m['reason']
        }
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f"✅ Gerado: {clip_name}.mp4")

# 4. Merge Final
final_path = os.path.join(output_dir, "compilacao_final.mp4")
editor.merge_videos(clip_paths, final_path)
print(f"✨ Processo Finalizado! Seus vídeos estão na pasta: {output_dir}")